# 18 — Triggering a CellReg run from the database

This notebook plans, exports, runs and ingests a **new** cross-session cell
registration. It is the opposite direction from notebook 17, which parses
registrations that already exist on disk.

The point of doing it this way is one specific fix. Leon's historical exports copied
`<ScanPath>/suite2p/plane0/Fall.mat` — a per-scan **filesystem path with no curation in
the lookup** — so which curation a column came from was never an input, and cannot be
recovered afterwards: of 743 `(session, scan, paramset)` groups holding more than one
curation, 696 (94 %) have byte-identical mask sets, so mask-set overlap scores 1.0
against every candidate and discriminates nothing. Lab-wide that cost us: of the 1,266
columns in the 101 historical runs, only **565** could be attached to a curation at all, and
of those only 317 by content evidence.

`CellRegTask.Session` removes the problem at the source. The operator **declares** the
curation for each column *before* anything runs, the exporter writes that manifest
beside the footprints, and the importer reads it back as
`resolution_method='manifest'` — a recorded fact rather than a reconstruction.

---
**Every write in this notebook is behind `RUN_WRITES`, which defaults to `False`.**
Read it top to bottom with writes off; set it to `True` only for a site you intend to
register. Nothing here ever writes `Segmentation.Mask.roi_group`.

In [ ]:
RUN_WRITES = False   # set True to actually create a task, export and ingest

import os
import datajoint as dj

# dj_local_conf.json carries custom.database.prefix, which adamacs reads AT IMPORT.
# Load it before importing the pipeline or the schema names come out wrong.
dj.config.load(os.path.expanduser('~/adamacs/dj_local_conf.json'))

from adamacs.pipeline import (cell_registration as cr, imaging, scan, session,
                              subject)
print('cell_registration schema:', cr.CellRegRun.database)

## 1. Choose the site and the sessions

A *site* is one field of view followed over days. `RegistrationSite.same_site_id` is the
site's **first** session id, matching the cellreg folder convention
(`LE_ROS-1881_sess9FPXNI8U_Gratings_CR`) and `roselab_session.session_same_site`.

Upstream `same_site_id` cannot be used as a foreign key yet: it is populated for all
3,460 sessions but **defaults to self-reference** — 2,561 have
`same_site_id = session_id` — so "never assigned to a site" and "a site of exactly one
session" are indistinguishable there. Hence a local table, with `site_id_source`
recording which case a row came from.

In [ ]:
SUBJECT  = 'ROS-1881'
SITE     = 'sess9FPXNI8U'      # the site's first session id
STIMULUS = 'Gratings'          # one run per stimulus type per site

site_key = {'subject': SUBJECT, 'same_site_id': SITE}

# Restrict to THIS SITE, not just this subject. roselab_session.session_same_site maps
# session_id -> same_site_id, so it is the authoritative site membership even though it
# cannot express 'unknown' (see RegistrationSite's docstring). A subject-only query would
# sweep in every other field of view the animal was imaged at, and auto_generate_entries
# only checks that curations EXIST -- it cannot tell that two of them are different FOVs.
at_site = session.SessionSameSite & {'same_site_id': SITE}

# Stimulus has no structured field: scan.Scan carries only free-text scan_notes, so the
# operator picks the scans. Print them with their notes to choose from.
candidates = (imaging.Curation * scan.Scan) & at_site
print(f'{len(candidates)} curations at site {SITE}')
candidates

### Declare the column order explicitly

The **order** of this list is the column order of the run, and it is recorded, never
inferred. Pick exactly one curation per scan — if a scan has several, this is the
moment the choice is made and written down, which is precisely what the historical runs
failed to do.

In [ ]:
# Name the scans for THIS stimulus explicitly, in the column order you want. There is no
# database field that says 'Gratings', so this list is the operator's declaration -- which
# is the whole point of the task table.
SCANS = [
    # 'scan9FPZGABI', 'scan9FQ0MLVT', ...   # paste from the table above
]

curations = [dict(c) for c in
             (imaging.Curation & at_site & [{'scan_id': s} for s in SCANS]
              & 'curation_id = 0').fetch('KEY', order_by='scan_id')] if SCANS else []

# Guards, because a wrong column set is silent until the registration is meaningless:
site_sessions = set((at_site).fetch('session_id'))
assert all(c['session_id'] in site_sessions for c in curations), \
    'a declared curation is not at this site'
assert len(curations) == len(SCANS), \
    f'{len(SCANS)} scans named but {len(curations)} curations found -- check curation_id'
assert len({c['scan_id'] for c in curations}) == len(curations), \
    'the same scan appears twice'

for i, c in enumerate(curations):
    print(i, c['session_id'], c['scan_id'], 'paramset', c['paramset_idx'],
          'curation', c['curation_id'])
print(f'\n{len(curations)} columns declared')

## 2. Register the parameter set

`CellRegParamSet` is **immutable by content**: re-registering a slug with different
parameters raises rather than updating, so a changed parameter is a new slug and every
downstream key names the parameters that produced it. The slug carries the first 8 hex
of the content hash — without it, runs that share approach, alignment and `p_same`
threshold but differ in `maximal_distance_um` or `n_bins` collide (86 of the 101
historical runs did exactly that).

In [ ]:
params = {
    'registration_approach': 'probabilistic',
    'alignment_mode': 'translations_and_rotations',
    'model_type': 'Spatial correlations',
    'initial_registration_type': 'Spatial correlations',
    'initial_threshold': 0.5,
    'final_threshold': 0.5,
    'maximal_distance_um': 12.0,
    'n_bins': 51,
}

from adamacs.ingest.cellreg_register import paramset_for
slug, method, _ = paramset_for({'registration_approach': 'Probabilistic modeling',
                                'alignment_type': 'Translations and Rotations',
                                'final_threshold': 0.5, **params})
print('slug:', slug, '| matching_method:', method)

if RUN_WRITES:
    cr.CellRegParamSet.register(slug, params, label=f'{STIMULUS} {SITE}',
                                matching_method=method)
cr.CellRegParamSet()

## 3. Plan the task

`auto_generate_entries` writes **nothing to disk** — it only plans. It refuses a run of
fewer than two sessions (registering one session against itself is not a registration),
refuses a duplicated curation in the column order, and refuses curations that do not
exist.

`task_mode` follows the house convention (`imaging.ProcessingTask`, `wfield.py:372`,
`disk.py:986`):

- `'trigger'` — build the CellReg input files from the database for the declared
  curations, then run the MATLAB driver over that directory
- `'load'` — a run already exists in `cellreg_output_dir`; record what is there

In [ ]:
if RUN_WRITES:
    if not (cr.RegistrationSite & site_key):
        cr.RegistrationSite.insert1({**site_key,
                                     'site_description': f'{SUBJECT} {STIMULUS}',
                                     'site_id_source': 'cellreg_folder',
                                     'created': __import__('datetime').datetime.now()})
    task_key = cr.CellRegTask.auto_generate_entries(
        site_key, curations, paramset=slug, task_mode='trigger',
        label=STIMULUS, created_by=os.environ.get('USER', ''),
        note='planned from notebook 18')
    print('task:', task_key)
else:
    task_key = None
    print('RUN_WRITES is False -- no task created')

cr.CellRegTask()

In [ ]:
# THE MANIFEST: one row per column, naming the curation it is built from.
# This is what makes resolution_method='manifest' possible on the way back in.
cr.CellRegTask.Session & (task_key or {})

## 4. Export the inputs from the database

`CellRegExport` is a `dj.Imported` table, so the export runs through `populate()`. It
writes one `converted_Fall_NN_<scan_id>.mat` per column into the task's flat
`cellreg_output_dir`, built **from `Segmentation.Mask`**, not by copying a `Fall.mat`
off disk.

That encoder is verified: the MATLAB driver reads only `ops.Lx`, `ops.Ly`,
`size(stat,2)` and each `stat{k}.ypix / xpix / lam` — it does not even read `iscell` —
and a database-built file reproduced all 11 scans of `sess9FPXNI8U` exactly, 13,532
ROIs. Two consequences worth knowing:

- the driver pads to `ops.Lx + 1`, so the log reports a **513**-pixel image for a
  512-pixel scan and `microns_per_pixel` is 615.6/513 = **1.2**, not 615.6/512 = 1.2024;
  that factor scales every centroid distance through `normalized_maximal_distance`
- the manifest `(column_index, session_id, scan_id, paramset_idx, curation_id)` is
  written beside the footprints — this is the whole point of the exercise

In [ ]:
if RUN_WRITES and task_key is not None:
    cr.CellRegExport.populate(task_key, display_progress=True)
    print((cr.CellRegExport & task_key).fetch1())

cr.CellRegExport.File & (task_key or {})

## 5. Run CellReg on ibehavegpu1

CellReg is MATLAB and **must not run on tatchu3** — it exhausted RAM there
(2025-02-20) and Tobias ruled it out (2025-08-26). It runs under `backup_user` on
ibehavegpu1:

| | |
|---|---|
| MATLAB | `$HOME/MATLAB/bin/matlab`, R2023a Update 8 (9.14.0.2891782) |
| CellReg | `$HOME/CellReg`, commit `70668513c665` (v1.6.0) |
| Worker | `scripts/cellreg_worker_ibehavegpu.sh` |

Leon's install at `/home/leonk/CellReg` is **not** used: `/home/leonk` is mode 700 and
his licence is tied to his username.

The worker polls for `task_mode='trigger'` tasks, exports what is missing, calls the
MATLAB driver, and ingests the result. Its dry run reports:

```
host=ibehavegpu1 toolbox=/home/backup_user/CellReg sha=70668513c665
matlab=...version=9.14.0.2891782 (R2023a) Update 8
attached read-only to roselab_cell_registration; no table can be created on this path
pass complete: {"exported": 0, "matlab": [], "ingested": 0}
```

```bash
ssh ibehavegpu1
sudo -u backup_user -i
cd ~/adamacs_ingest
./scripts/cellreg_worker_ibehavegpu.sh --once              # DRY RUN: reports, writes nothing
./scripts/cellreg_worker_ibehavegpu.sh --execute --once     # one real pass -- WRITES
```

**There is no `--dry-run` flag** — the wrapper forwards its arguments to
`cellreg_worker.py`, whose options are `--schema --matlab --cellreg-root --poll-seconds
--matlab-timeout --once --execute --force-host --skip-matlab-check`, so `--dry-run` exits
with an argument error. Dry run is the **default**: without `--execute` the worker reports
what it would do and neither writes to the database nor starts MATLAB. That means `--once`
alone is a verification pass, and a real run needs `--execute`.

One settings trap, caught by a smoke test and now guarded by `SETTINGS_CHOICES`:
the driver's literals must match the log's prose exactly — `'Probabilistic modeling'`
and `'Spatial correlations'` (plural). `strcmp` against `'Probabilistic'` or
`'Spatial correlation'` fails **silently** after stage 5 and produces no result file.

## 6. Ingest the finished run

Registration is content-addressed: `cellreg_run_id` is the first 40 hex of the result
file's sha256, so byte-identical backups scattered across scan folders collapse to one
run with its other paths in `CellRegRun.Copy` (108 historical paths → 100 distinct
runs). Because this run came from a task, its columns resolve from the manifest.

In [ ]:
if RUN_WRITES and task_key is not None:
    import glob
    import json
    import datetime
    from adamacs.ingest import cellreg as CR
    from adamacs.ingest import cellreg_register as REG

    out_dir = (cr.CellRegTask & task_key).fetch1('cellreg_output_dir')
    results = sorted(glob.glob(os.path.join(out_dir, 'Results',
                                            'cellRegistered_*.mat')))
    print('result files:', results)

    # REGISTER FIRST. CellRegResult is keyed on CellRegRun, so populate() has nothing to
    # do until the run exists -- and register_run is what reads the manifest: it finds the
    # task whose cellreg_output_dir contains this file, takes the declared curation for
    # each column from CellRegTask.Session, and records resolution_method='manifest'.
    # A centroid count that disagrees with a declared curation is a hard error here, not a
    # demotion: the exporter built the column from that curation's masks, so a mismatch
    # means the directory and the task describe different exports.
    for path in results:
        rec = {'canonical': path, 'run_id': CR.file_sha256(path)[:40],
               'sha256': CR.file_sha256(path),
               'bytes': os.stat(path).st_size,
               'mtime': datetime.datetime.fromtimestamp(
                   os.stat(path).st_mtime).isoformat(),
               'mat_format': CR.detect_mat_format(path),
               'log_file': CR.find_log_file(path), 'copies': [],
               'folder': os.path.basename(out_dir)}
        plan = REG.register_run(cr, rec, cr.connection, dry_run=False,
                                census=REG.curation_census(cr.connection))
        print(f"  {os.path.basename(path)}: {plan['n_resolved']} resolved, "
              f"{plan['n_unresolved']} unresolved")

    # now the parsed content follows from the registered run
    cr.CellRegResult.populate(display_progress=True)
    cr.MaskCoordinates.populate(display_progress=True)

    print('\nresolution_method:',
          dict(__import__('collections').Counter(
              cr.CellRegRun.Session.fetch('resolution_method'))))

## 6b. Why the manifest matters — what run 1 measured

Run 1 registered all 101 historical runs on 2026-10-20. Those runs predate
`CellRegTask`, so none of them has a manifest, and the result is the argument for this
notebook:

| Columns | Outcome |
|---:|---|
| 317 | `roi_group_match` — a unique exact content match |
| 537 | `mask_set_identical` — candidates matched the centroid count *and* were byte-identical |
| 248 | `log_file` — the scan has exactly one curation, so forced |
| 50 | no curation's mask count matches the file at all |
| 25 | candidates match the count but hold **different** mask sets — left unresolved |
| 65 | empty column |
| 24 | no scan id recoverable from the log |
| **1,266** | total, = `SUM(n_sessions)` over the 101 runs |

1,102 resolved, 164 not. That took the memberships reachable from a curation from 46.6 % to
**94.6 %** — but 537 of those 1,102 rest on a *convention* (the candidates were provably the
same mask set, so the choice could not change the data) rather than on evidence about which
curation was actually used.

None of that is a manifest. For 537 columns we can say *these masks*, but not *this
curation was the declared input* — and for 25 we cannot even say that. `roi_group` was
overwritten in place, so only the last run to target a curation is reflected there; the
registrations themselves survive in full, but the link back to a declared input does not.

### Two lessons that change how you should read `resolution_method`

**`log_file` is not evidence.** It means only one candidate curation existed for that
scan — not that it is the right one. Eight of run 1's columns were attached that way and
then **refused by `MaskCoordinates`**, whose guard compares
`centroid_locations_corrected` row counts against the curation's mask count:

| | |
|---|---|
| off-by-one | 1034 vs 1035, 1280 vs 1279, 1115 vs 1116 — the curation was re-curated after the run |
| wholesale | 316 vs 910, 370 vs 1088, 336 vs 1064 — the input file is a different segmentation |

They are now in `UnresolvedColumn` with both counts recorded, and
`cellreg_register.centroid_counts()` performs the same comparison before a `Session` row
is ever written. A task-driven run does not face this: the manifest names the curation,
and the exporter builds the input **from that curation's masks**, so the centroid count
cannot disagree.

**A tie is not resolvable by content.** Where candidate curations are byte-identical in
both masks *and* `roi_group`, no analysis separates them. A convention was declared for
exactly that case on 2026-10-20 — `mask_set_identical`, which attached 537 columns — and it
is defensible *only* because the candidates are provably the same mask set, so the choice
cannot change which masks, centroids or traces the column resolves to. Where the candidates
genuinely differ (25 columns) nothing was attached: that would be choosing between
different answers. Even for the 537, what the convention supplies is a `curation_id`, not
knowledge of which curation the run was actually launched from. Going forward the manifest
removes the question entirely.

### Pair quality is stored, and its gaps are counted

`CellRegResult.PairQuality` holds 4,641,778 rows over the 101 runs — one per
`(cell, column_a, column_b)` where the cell is present in **both** columns. It is
legitimately shorter than the index maps imply, because CellReg emits `NaN` `p_same` for
some pairs it nonetheless calls the same cell: **4,807,688 expected vs 4,641,778 stored**,
recorded per run in `n_pairs_expected` / `n_pairs_nonfinite` so the difference can never be
mistaken for a parser fault. Of the 165,910 gap, 159,940 are confirmed non-finite; the
remaining 5,970 belong to one run whose `p_same` array could not be read at all, where
`n_pairs_nonfinite` is NULL. The field counts values that were read and found non-finite —
never "absent", which is a different question.

## 7. What is deliberately *not* written

`Segmentation.Mask.roi_group` is **never written by any of this**. It stays a derived
projection in `RoiGroupProjection`, which carries the `cell_index` that explains each
value, next to the run that produced it.

That is the whole reason this schema exists. `roi_group` is a `smallint` outside the
primary key that was overwritten in place by each successive run, so only the last run
to target a curation survived: of the 1,266 historical columns, **533 match no
curation's `roi_group` at all** — those identities are simply gone. Writing a new run's
identities back into the same column would destroy the previous one the same way.

If you ever do want the column refreshed, that is an explicit, separately authorised
step with `RoiGroupSnapshot` taken first — not a side effect of registering a run.

In [ ]:
# proof, any time you want it: nothing here populates these
print('RoiGroupProjection rows:', len(cr.RoiGroupProjection()))
print('RoiGroupSnapshot  rows:', len(cr.RoiGroupSnapshot()))